# QData Free Source Factor Backtest

This notebook is a concise walkthrough of a tiny factor research workflow on the QData mock backend. It does not require Docker, paid data, pandas, or external network access.

The demo ranks the tradable HS300 mock universe by `momentum_20d` on `2024-01-02`, buys the top-ranked stock for one day, and compares it with an equal-weight benchmark.

In [1]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if not (repo_root / "qdata").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from qdata import Client
from examples.factor_backtest_demo import run_demo, format_report

## 1. Build a tradable universe

A quant research database should not blindly rank every listed security. The first step is to apply tradability filters such as suspension, ST status, delisting period and minimum listing days.

In [2]:
client = Client(default_format="records")
tradable = client.get_tradable_universe(
    asof_date="2024-01-02",
    universe="hs300",
    min_list_days=120,
)
tradable

[{'symbol': '600519.SH',
  'security_id': 1000001,
  'asof_date': '2024-01-02',
  'can_buy': True,
  'can_sell': True,
  'list_days': 8163,
  'is_st': False,
  'is_suspended': False,
  'is_new_listing': False,
  'is_delisting_period': False},
 {'symbol': '000001.SZ',
  'security_id': 1000002,
  'asof_date': '2024-01-02',
  'can_buy': True,
  'can_sell': True,
  'list_days': 11962,
  'is_st': False,
  'is_suspended': False,
  'is_new_listing': False,
  'is_delisting_period': False}]

## 2. Pull point-in-time factor values

The factor signal is requested as of the signal date. In a real setup, this prevents future data leakage.

In [3]:
symbols = [row["symbol"] for row in tradable]
client.get_factor(
    factors=["momentum_20d", "roe_ttm"],
    symbols=symbols,
    start_date="2024-01-02",
    end_date="2024-01-02",
    format="wide",
)

[{'symbol': '600519.SH',
  'security_id': 1000001,
  'trade_date': '2024-01-02',
  'momentum_20d': 0.032,
  'roe_ttm': 0.283},
 {'symbol': '000001.SZ',
  'security_id': 1000002,
  'trade_date': '2024-01-02',
  'momentum_20d': -0.011,
  'roe_ttm': 0.104}]

## 3. Run the tiny backtest

The script version lives in `examples/factor_backtest_demo.py`. It uses only the Python standard library plus the QData SDK.

In [4]:
result = run_demo()
print(format_report(result))

QData factor backtest demo
universe=hs300 factor=momentum_20d signal_date=2024-01-02 hold_date=2024-01-03
tradable_symbols=2 long_bucket=600519.SH short_bucket=000001.SZ
long_return=0.4122% benchmark_return=0.6237% active_return=-0.2114% factor_spread=-0.4228%


## 4. What this demonstrates

- The research workflow starts from a tradable universe instead of raw symbols.
- Factor values are pulled by signal date to avoid future leakage.
- Prices are forward-adjusted through the SDK.
- The result reports strategy return, benchmark return, active return and factor spread.

The dataset is intentionally tiny so the notebook is easy to review as a technical walkthrough. The same interface can be pointed at the SQL backend after the local PostgreSQL and ClickHouse stack is running.